In [1]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"Device          : {device}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")


def print_gpu_memory(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA unavailable")
        return

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 3)
    free, total = torch.cuda.mem_get_info()

    free = free / (1024 ** 3)
    total = total / (1024 ** 3)

    print(
        f"[{tag}] "
        f"allocated={allocated:.2f} GiB | "
        f"reserved={reserved:.2f} GiB | "
        f"peak={max_allocated:.2f} GiB | "
        f"free={free:.2f} GiB / {total:.2f} GiB"
    )


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print_gpu_memory("initial")

PyTorch version : 2.10.0+cu128
CUDA available  : True
Device          : cuda
GPU             : Tesla T4
CUDA version    : 12.8
[initial] allocated=0.00 GiB | reserved=0.00 GiB | peak=0.00 GiB | free=14.46 GiB / 14.56 GiB


In [2]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m","pip","install","-q","--no-deps","torchao>=0.16.0"])
!pip install -q liger-kernel
!pip install traker[fast]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.3/644.3 kB 31.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for fast_jl: filename=fast_jl-0.1.3-cp312-cp312-linux_x86_64.whl size=958599 sha256=b16bf1bbd6804ec5b0cd0a297a6e357b6b614328bd8fdb96908a79eecedc4663
  Stored in directory: /root/.cache/pip/wheels/cd/5a/bd/a05a64ea6e542809cfafc036bcbd44f0c20a04115917e3ab0f
  Created wheel for traker: filename=traker-0.3.2-py3-none-any.whl size=29027 sha256=2da710d2e91279704337dc4c679dbbf03480160d94421bf4c59aa89697626b21
  Stored in directory: /root/.cache/pip/wheels/6f/9c/93/cacba5ebe6989142debfcab420506346a9b9d6cd7bae112161
Successfully built fast_jl traker


In [3]:
import os
import gc
import json
import random
from pathlib import Path

import pandas as pd
import numpy as np

from datasets import load_dataset, concatenate_datasets
from datasets import load_from_disk

from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from transformers import (AutoModelForCausalLM, AutoTokenizer)

from peft import PeftModel

from trak.projectors import (BasicProjector, CudaProjector, ProjectionType)
from liger_kernel.transformers import (LigerFusedLinearCrossEntropyLoss)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 2048

MODEL_DTYPE = torch.bfloat16

LORA_R = 128
LORA_ALPHA = 512
LORA_DROPOUT = 0.10

LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

NUM_CHECKPOINTS = 4
GRADIENT_BATCH_SIZE = 1

PROJECTION_BATCH_SIZE = 4
PROJECTION_SEED = 0

ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

NUM_CANDIDATES = 9000

In [5]:
def set_seed(seed: int):  
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("after runtime setup")

CUDA device: Tesla T4
[after runtime setup] allocated=0.00 GiB | reserved=0.00 GiB | peak=0.00 GiB | free=14.46 GiB / 14.56 GiB


In [6]:
CHECKPOINT_ROOT = Path("/kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints")
DATASET_PATH = Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw")
OUTPUT_ROOT = Path("/kaggle/working/less_phase2")

print("Checkpoint root:", CHECKPOINT_ROOT)
print("Dataset path   :", DATASET_PATH)
print("Output root    :", OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

checkpoint_dirs = sorted(
    [
        p for p in CHECKPOINT_ROOT.iterdir()
        if p.is_dir()
    ]
)

print("\nDiscovered checkpoint directories:")

for p in checkpoint_dirs:
    print(f"  {p.name}")

if len(checkpoint_dirs) != NUM_CHECKPOINTS:
    raise RuntimeError(
        f"Expected {NUM_CHECKPOINTS} checkpoints, "
        f"but found {len(checkpoint_dirs)}:\n"
        + "\n".join(f"  {p}" for p in checkpoint_dirs)
    )



for ckpt_dir in checkpoint_dirs:
    print("=" * 80)
    print(f"CHECKPOINT: {ckpt_dir.name}")
    print("=" * 80)

    files = sorted(
        p for p in ckpt_dir.rglob("*")
        if p.is_file()
    )

    if not files:
        print("  WARNING: checkpoint is empty")
        continue

    total_size = 0

    for file_path in files:
        size_mb = file_path.stat().st_size / (1024 ** 2)
        total_size += size_mb

        relative_path = file_path.relative_to(ckpt_dir)

        print(f"  {str(relative_path):45s} {size_mb:10.2f} MB")

    print(f"\n  Total checkpoint size: {total_size:.2f} MB")
    print()

Checkpoint root: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints
Dataset path   : /kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw
Output root    : /kaggle/working/less_phase2

Discovered checkpoint directories:
  epoch_1
  epoch_2
  epoch_3
  epoch_4
CHECKPOINT: epoch_1
  README.md                                           0.00 MB
  adapter_config.json                                 0.00 MB
  adapter_model.safetensors                         133.03 MB
  chat_template.jinja                                 0.00 MB
  tokenizer.json                                     10.89 MB
  tokenizer_config.json                               0.00 MB
  training_state.pt                                 266.18 MB

  Total checkpoint size: 410.11 MB

CHECKPOINT: epoch_2
  README.md                                           0.00 MB
  adapter_config.json                                 0.00 MB
  adapter_model.safetensors                         133.03 MB
  chat_temp

In [7]:
import shutil

RESUME_MARKER = "progress_state.json"
resume_source = None

for candidate_root in Path("/kaggle/input").glob("*"):
    for marker_path in candidate_root.rglob(RESUME_MARKER):
        # marker_path looks like .../less_phase2/gradient_datastore/progress_state.json
        candidate_phase2_dir = marker_path.parent.parent
        if candidate_phase2_dir.name == "less_phase2" or (candidate_phase2_dir / "gradient_datastore").exists():
            resume_source = candidate_phase2_dir
            break
    if resume_source is not None:
        break

if resume_source is not None:
    print(f"Found previous session's output: {resume_source}")
    print(f"Copying into {OUTPUT_ROOT} ...")

    shutil.copytree(resume_source, OUTPUT_ROOT, dirs_exist_ok=True)

    copied_files = sorted(OUTPUT_ROOT.rglob("*"))
    print(f"Copied {len(copied_files)} files/dirs. Resume data is in place.")
else:
    print(
        "No previous committed output found under /kaggle/input -- "
        "starting fresh. (This is expected on the very first session, or "
        "if you haven't attached last session's committed output yet.)"
    )


Found previous session's output: /kaggle/input/notebooks/manasaindusrikarri/phase2-gradient-computation-epoch-3/less_phase2
Copying into /kaggle/working/less_phase2 ...
Copied 6 files/dirs. Resume data is in place.


In [8]:
ckpt = checkpoint_dirs[0]

training_state_path = ckpt / "training_state.pt"
state = torch.load(
    training_state_path,
    map_location="cpu",
    weights_only=False,
)

optimizer_state_dict = state["optimizer"]

optimizer_state = optimizer_state_dict["state"]
optimizer_param_groups = optimizer_state_dict["param_groups"]

print(f"Number of optimizer states : {len(optimizer_state)}")
print(f"Number of parameter groups : {len(optimizer_param_groups)}")
\
print("\nFirst 5 optimizer state entries:\n")

for i, (param_id, param_state) in enumerate(optimizer_state.items()):

    print(f"Parameter ID: {param_id}")
    print(f"State keys  : {list(param_state.keys())}")

    for key, value in param_state.items():

        if torch.is_tensor(value):
            print(
                f"  {key:12s} "
                f"shape={tuple(value.shape)!s:25s} "
                f"dtype={value.dtype} "
                f"device={value.device}"
            )
        else:
            print(
                f"  {key:12s} "
                f"type={type(value).__name__} "
                f"value={value}"
            )

    print()

    if i >= 4:
        break

Number of optimizer states : 224
Number of parameter groups : 1

First 5 optimizer state entries:

Parameter ID: 0
State keys  : ['step', 'exp_avg', 'exp_avg_sq']
  step         shape=()                        dtype=torch.float32 device=cpu
  exp_avg      shape=(128, 1536)               dtype=torch.float32 device=cpu
  exp_avg_sq   shape=(128, 1536)               dtype=torch.float32 device=cpu

Parameter ID: 1
State keys  : ['step', 'exp_avg', 'exp_avg_sq']
  step         shape=()                        dtype=torch.float32 device=cpu
  exp_avg      shape=(1536, 128)               dtype=torch.float32 device=cpu
  exp_avg_sq   shape=(1536, 128)               dtype=torch.float32 device=cpu

Parameter ID: 2
State keys  : ['step', 'exp_avg', 'exp_avg_sq']
  step         shape=()                        dtype=torch.float32 device=cpu
  exp_avg      shape=(128, 1536)               dtype=torch.float32 device=cpu
  exp_avg_sq   shape=(128, 1536)               dtype=torch.float32 device=cpu

Para

In [9]:
# Load Base Model
print_gpu_memory("before base model")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    device_map={"": device},
)

base_model.eval()

print("\nBase model loaded.")
print(f"Model device: {next(base_model.parameters()).device}")
print(f"Model dtype : {next(base_model.parameters()).dtype}")

print_gpu_memory("after base model")

[before base model] allocated=0.00 GiB | reserved=0.00 GiB | peak=0.00 GiB | free=14.46 GiB / 14.56 GiB


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Base model loaded.
Model device: cuda:0
Model dtype : torch.bfloat16
[after base model] allocated=2.88 GiB | reserved=2.93 GiB | peak=2.88 GiB | free=11.53 GiB / 14.56 GiB


In [10]:
# Load LoRA Checkpoint
checkpoint = checkpoint_dirs[0]

print(f"Loading LoRA checkpoint: {checkpoint}")
print_gpu_memory("before LoRA")


model = PeftModel.from_pretrained(
    base_model,
    checkpoint,
    is_trainable=True,
)

print("\nLoRA checkpoint loaded.")


print_gpu_memory("after LoRA")

trainable_params = 0
total_params = 0

for param in model.parameters():
    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(
    f"Trainable percentage : "
    f"{100 * trainable_params / total_params:.4f}%"
)

Loading LoRA checkpoint: /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints/epoch_1
[before LoRA] allocated=2.88 GiB | reserved=2.93 GiB | peak=2.88 GiB | free=11.53 GiB / 14.56 GiB

LoRA checkpoint loaded.
[after LoRA] allocated=3.01 GiB | reserved=3.26 GiB | peak=3.14 GiB | free=11.18 GiB / 14.56 GiB

Total parameters     : 1,578,579,456
Trainable parameters : 34,865,152
Trainable percentage : 2.2086%


In [11]:
trainable_parameters = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

optimizer_entries = list(optimizer_state.items())

if len(trainable_parameters) != len(optimizer_entries):
    raise RuntimeError(
        "Number of trainable parameters does not match "
        "number of optimizer states."
    )


mapping_errors = []

for index, ((name, param), (param_id, param_state)) in enumerate(
    zip(trainable_parameters, optimizer_entries)
):

    exp_avg = param_state["exp_avg"]
    exp_avg_sq = param_state["exp_avg_sq"]

    if tuple(param.shape) != tuple(exp_avg.shape):
        mapping_errors.append(
            {
                "index": index,
                "parameter": name,
                "param_id": param_id,
                "issue": "exp_avg shape mismatch",
                "model_shape": tuple(param.shape),
                "state_shape": tuple(exp_avg.shape),
            }
        )

    if tuple(param.shape) != tuple(exp_avg_sq.shape):
        mapping_errors.append(
            {
                "index": index,
                "parameter": name,
                "param_id": param_id,
                "issue": "exp_avg_sq shape mismatch",
                "model_shape": tuple(param.shape),
                "state_shape": tuple(exp_avg_sq.shape),
            }
        )


print(f"Parameters checked : {len(trainable_parameters)}")
print(f"Mapping errors     : {len(mapping_errors)}")

if mapping_errors:
    print("\nFirst mapping errors:")

    for error in mapping_errors[:10]:
        print(error)

    raise RuntimeError(
        "Parameter ↔ optimizer mapping validation failed."
    )

print("\n All 224 model parameters match optimizer state shapes.")
print("Positional parameter/state mapping is structurally valid.")

Parameters checked : 224
Mapping errors     : 0

 All 224 model parameters match optimizer state shapes.
Positional parameter/state mapping is structurally valid.


In [12]:
from datasets import load_dataset, concatenate_datasets

DATA_FILES = {
    "flan_v2": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/flan_v2_mini.jsonl"),
    "cot": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/cot_mini.jsonl"),
    "dolly": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/dolly_mini.jsonl"),
    "oasst1": Path("/kaggle/input/notebooks/tanmairaghava/data-preprocessing/data/raw/oasst1_mini.jsonl"),
}

datasets = {}

for name, path in DATA_FILES.items():
    ds = load_dataset(
        "json",
        data_files=str(path),
        split="train",
    )
    datasets[name] = ds
    print(f"{name:10s}: {len(ds):} entries")


def merge_datasets(dataset_dict):
    return concatenate_datasets(list(dataset_dict.values()))


candidate_dataset = merge_datasets(datasets)
print(f"\nMerged dataset: {len(candidate_dataset):} entries")


example = candidate_dataset[0]

print("\nFirst example:")
print(example)

Generating train split: 0 examples [00:00, ? examples/s]

flan_v2   : 3000 entries


Generating train split: 0 examples [00:00, ? examples/s]

cot       : 3000 entries


Generating train split: 0 examples [00:00, ? examples/s]

dolly     : 1500 entries


Generating train split: 0 examples [00:00, ? examples/s]

oasst1    : 1500 entries

Merged dataset: 9000 entries

First example:
{'dataset': 'flan_v2', 'id': 'flan_v2_83810', 'messages': [{'role': 'user', 'content': "In this task, you need to count the number of words in a sentence that contain the given letter\n\nExample input: Sentence: 'two gray birds standing in a green field'. How many words contain the letter 'a' in the sentence.\nExample output: 3\nExample explanation: The words 'gray', 'standing', and 'a' contain the letter 'a'. So, the answer is 3.\nQ: Sentence: 'a woman in a red shirt in the kitchen of a house with ceiling fan light on in the room'. How many words contain the letter 'l' in the sentence.\nA:"}, {'role': 'assistant', 'content': '2'}]}


In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"PAD token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")


def tokenize_example(example):
    encoded = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )

    if hasattr(encoded, "encodings") and encoded.encodings:
        input_ids = encoded.encodings[0].ids

    elif hasattr(encoded, "input_ids"):
        input_ids = encoded.input_ids

        if input_ids and isinstance(input_ids[0], list):
            input_ids = input_ids[0]

    else:
        input_ids = encoded

    input_ids = list(input_ids)[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": input_ids.copy(),
        "attention_mask": [1] * len(input_ids),
    }


tokenized_candidate = candidate_dataset.map(
    tokenize_example,
    remove_columns=candidate_dataset.column_names,
    desc="Tokenizing 9,000 candidate examples",
)

print("Tokenized examples:", len(tokenized_candidate))
print("Columns:", tokenized_candidate.column_names)
print(
    "First example token length:",
    len(tokenized_candidate[0]["input_ids"])
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B
Vocabulary size: 151,665
PAD token: '<|endoftext|>'
EOS token: '<|endoftext|>'


Tokenizing 9,000 candidate examples:   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenized examples: 9000
Columns: ['input_ids', 'labels', 'attention_mask']
First example token length: 147


In [14]:
model.config.use_cache = False
model.base_model.model.config._attn_implementation = "eager"

model.enable_input_require_grads()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

from liger_kernel.transformers import (
    LigerFusedLinearCrossEntropyLoss
)

fused_loss_fn = LigerFusedLinearCrossEntropyLoss(
    reduction="mean",
    ignore_index=-100,
)

transformer = model.base_model.model.model
lm_head = model.base_model.model.lm_head

model.train()

print("Memory-efficient gradient path configured.")
print(f"use_cache                 : {model.config.use_cache}")
print(
    "attention implementation  :",
    model.base_model.model.config._attn_implementation,
)
print("gradient checkpointing     : enabled")
print("input require grads        : enabled")
print("Liger fused CE             : enabled")
print("model mode                 :", "train" if model.training else "eval")

print_gpu_memory("after memory configuration")

Memory-efficient gradient path configured.
use_cache                 : False
attention implementation  : eager
gradient checkpointing     : enabled
input require grads        : enabled
Liger fused CE             : enabled
model mode                 : train
[after memory configuration] allocated=3.01 GiB | reserved=3.26 GiB | peak=3.14 GiB | free=11.18 GiB / 14.56 GiB


In [15]:
lora_named_parameters = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

gradient_parameter_metadata = []

for parameter_index, (name, param) in enumerate(
    lora_named_parameters
):

    gradient_parameter_metadata.append(
        {
            "index": parameter_index,
            "name": name,
            "shape": tuple(param.shape),
            "numel": param.numel(),
            "dtype": str(param.dtype),
        }
    )


total_gradient_dimensions = sum(
    item["numel"]
    for item in gradient_parameter_metadata
)

print(
    f"Total raw gradient dimensions : "
    f"{total_gradient_dimensions:,}"
)

print("\nFirst 10 parameter entries:")

for item in gradient_parameter_metadata[:10]:
    print(
        f"[{item['index']:3d}] "
        f"{item['name']} "
        f"shape={item['shape']} "
        f"numel={item['numel']:,}"
    )

Total raw gradient dimensions : 34,865,152

First 10 parameter entries:
[  0] base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight shape=(128, 1536) numel=196,608
[  1] base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight shape=(1536, 128) numel=196,608
[  2] base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight shape=(128, 1536) numel=196,608
[  3] base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight shape=(256, 128) numel=32,768
[  4] base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight shape=(128, 1536) numel=196,608
[  5] base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight shape=(256, 128) numel=32,768
[  6] base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight shape=(128, 1536) numel=196,608
[  7] base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight shape=(1536, 128) numel=196,608
[  8] base_model.model.model.layers.1.self_attn.q_proj.lora_

In [16]:
GRADIENT_TYPE = "adam"

GRADIENT_DIMS = 8192

NUM_CHECKPOINTS = len(checkpoint_dirs)
NUM_CANDIDATES = len(candidate_dataset)

EXPECTED_TRAINABLE_TENSORS = 224
EXPECTED_RAW_GRADIENT_DIMS = 34_865_152


In [17]:
TRAINING_STATE_NAME = "training_state.pt"

print("Loading Adam optimizer states...")
print()

adam_states = {}

for checkpoint_dir in checkpoint_dirs:

    checkpoint_name = checkpoint_dir.name
    state_path = checkpoint_dir / TRAINING_STATE_NAME

    print(f"Loading: {checkpoint_name}")

    training_state = torch.load(
        state_path,
        map_location="cpu",
        weights_only=False,
    )

    optimizer_state = training_state["optimizer"]
    state_dict = optimizer_state["state"]

    print(f"  Optimizer states : {len(state_dict)}")
    print(
        f"  Epoch            : "
        f"{training_state['epoch']}"
    )
    print(
        f"  Global step      : "
        f"{training_state['global_step']}"
    )

    adam_states[checkpoint_name] = state_dict

    del training_state

Loading Adam optimizer states...

Loading: epoch_1
  Optimizer states : 224
  Epoch            : 1
  Global step      : 4
Loading: epoch_2
  Optimizer states : 224
  Epoch            : 2
  Global step      : 8
Loading: epoch_3
  Optimizer states : 224
  Epoch            : 3
  Global step      : 12
Loading: epoch_4
  Optimizer states : 224
  Epoch            : 4
  Global step      : 16


In [18]:
trainable_parameters = [
    p for p in model.parameters()
    if p.requires_grad
]

assert len(trainable_parameters) == 224

print(
    f"Model trainable tensors : "
    f"{len(trainable_parameters)}"
)

reference_checkpoint = "epoch_1"

optimizer_param_ids = (
    torch.load(
        checkpoint_dirs[0] / TRAINING_STATE_NAME,
        map_location="cpu",
        weights_only=False,
    )["optimizer"]["param_groups"][0]["params"]
)

assert len(optimizer_param_ids) == len(trainable_parameters)

print(
    f"Optimizer parameter IDs : "
    f"{len(optimizer_param_ids)}"
)


parameter_state_mapping = []

named_parameters = dict(model.named_parameters())

for position, (parameter, parameter_id) in enumerate(
    zip(trainable_parameters, optimizer_param_ids)
):

    parameter_name = next(
        name
        for name, p in named_parameters.items()
        if p is parameter
    )

    parameter_state_mapping.append(
        {
            "position": position,
            "parameter": parameter,
            "parameter_id": parameter_id,
            "name": parameter_name,
            "shape": tuple(parameter.shape),
            "numel": parameter.numel(),
        }
    )

Model trainable tensors : 224
Optimizer parameter IDs : 224


In [19]:
def transform_gradient_adam(gradient, exp_avg, exp_avg_sq,):

    assert gradient.shape == exp_avg.shape
    assert gradient.shape == exp_avg_sq.shape

    m = exp_avg.to(
        device=gradient.device,
        dtype=gradient.dtype,
        non_blocking=True,
    )

    v = exp_avg_sq.to(
        device=gradient.device,
        dtype=gradient.dtype,
        non_blocking=True,
    )

    transformed = (
        ADAM_BETA1 * m
        + (1.0 - ADAM_BETA1) * gradient
    ) / torch.sqrt(
        ADAM_BETA2 * v
        + (1.0 - ADAM_BETA2) * gradient.square()
        + ADAM_EPS
    )

    return transformed

In [20]:
def build_adam_gradient_vector(model, parameter_state_mapping, adam_state_dict, device):
    
    flat_gradient = torch.empty(
        total_gradient_dimensions,
        dtype=torch.float16,
        device=device,
    )

    offset = 0

    with torch.no_grad():

        for entry in parameter_state_mapping:

            parameter = entry["parameter"]
            parameter_id = entry["parameter_id"]
            numel = entry["numel"]

            gradient = parameter.grad

            assert gradient is not None

            state = adam_state_dict[parameter_id]

            exp_avg = state["exp_avg"]
            exp_avg_sq = state["exp_avg_sq"]

            transformed = transform_gradient_adam(
                gradient,
                exp_avg,
                exp_avg_sq,
            )

            assert transformed.shape == parameter.shape
            assert torch.isfinite(transformed).all()

            flat_gradient[
                offset : offset + numel
            ].copy_(
                transformed.reshape(-1),
                non_blocking=True,
            )

            offset += numel

            del transformed
            del gradient

    assert offset == total_gradient_dimensions

    return flat_gradient


In [21]:
import gc
import torch
from peft import set_peft_model_state_dict
from safetensors.torch import load_file


def load_lora_checkpoint_weights(model, checkpoint_dir, device):

    checkpoint_dir = Path(checkpoint_dir)

    adapter_path = checkpoint_dir / "adapter_model.safetensors"

    if not adapter_path.exists():
        raise FileNotFoundError(
            f"LoRA adapter not found: {adapter_path}"
        )

    adapter_state = load_file(
        str(adapter_path),
        device="cpu",
    )


    load_result = set_peft_model_state_dict(
        model,
        adapter_state,
        adapter_name="default",
    )

    del adapter_state

    gc.collect()

    return load_result

In [22]:
EXTRACTION_BATCH_SIZE = 1
SAVE_EVERY = 100

DATASTORE_DTYPE = np.float32

from pathlib import Path
from trak.projectors import ProjectionType

PROJECTION_TYPE = ProjectionType.rademacher
PROJECTOR_SEED = 0
PROJECTOR_BATCH_SIZE = 16


GRADIENT_DATASTORE_ROOT = (
    OUTPUT_ROOT / "gradient_datastore"
)

GRADIENT_DATASTORE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

gradient_datastores = {}

for checkpoint_path in checkpoint_dirs:

    checkpoint_name = Path(checkpoint_path).name

    datastore_path = (
        GRADIENT_DATASTORE_ROOT
        / f"{checkpoint_name}.dat"
    )

    shape = (
        len(tokenized_candidate),
        GRADIENT_DIMS,
    )

    expected_bytes = (
        shape[0] * shape[1] * np.dtype(DATASTORE_DTYPE).itemsize
    )

    if datastore_path.exists() and datastore_path.stat().st_size == expected_bytes:
        datastore = np.memmap(
            datastore_path,
            dtype=DATASTORE_DTYPE,
            mode="r+",
            shape=shape,
        )
        print(
            f"Resumed {checkpoint_name}: found existing datastore "
            f"(shape={datastore.shape}), opened in r+ mode -- data preserved. "
            f"file={datastore_path}"
        )
    else:
        datastore = np.memmap(
            datastore_path,
            dtype=DATASTORE_DTYPE,
            mode="w+",
            shape=shape,
        )

        datastore[:] = 0.0
        datastore.flush()

        print(
            f"Created {checkpoint_name}: "
            f"shape={datastore.shape}, "
            f"dtype={datastore.dtype}, "
            f"file={datastore_path}"
        )

    gradient_datastores[checkpoint_name] = datastore

Resumed epoch_1: found existing datastore (shape=(9000, 8192)), opened in r+ mode -- data preserved. file=/kaggle/working/less_phase2/gradient_datastore/epoch_1.dat
Resumed epoch_2: found existing datastore (shape=(9000, 8192)), opened in r+ mode -- data preserved. file=/kaggle/working/less_phase2/gradient_datastore/epoch_2.dat
Resumed epoch_3: found existing datastore (shape=(9000, 8192)), opened in r+ mode -- data preserved. file=/kaggle/working/less_phase2/gradient_datastore/epoch_3.dat
Resumed epoch_4: found existing datastore (shape=(9000, 8192)), opened in r+ mode -- data preserved. file=/kaggle/working/less_phase2/gradient_datastore/epoch_4.dat


In [23]:
def extract_adam_gradient_vector(
    model,
    transformer,
    lm_head,
    fused_loss_fn,
    input_ids,
    attention_mask,
    parameter_state_mapping,
    adam_state_dict,
    device,
):

    model.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=MODEL_DTYPE,
    ):

        outputs = transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )

        hidden_states = outputs.last_hidden_state.reshape(
            -1,
            outputs.last_hidden_state.shape[-1],
        )

        labels = input_ids.reshape(-1)

        loss = fused_loss_fn(
            lm_head.weight,
            hidden_states,
            labels,
        )

    loss.backward()

    adam_gradient_vector = build_adam_gradient_vector(
        model=model,
        parameter_state_mapping=parameter_state_mapping,
        adam_state_dict=adam_state_dict,
        device=device,
    )

    del outputs
    del hidden_states
    del labels
    del loss
    
    with torch.no_grad():

        projected = projector.project(
            adam_gradient_vector.unsqueeze(0),
            model_id=0,
        )

    projected_cpu = (
        projected
        .squeeze(0)
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    del projected
    del adam_gradient_vector

    model.zero_grad(set_to_none=True)

    gc.collect()
    torch.cuda.empty_cache()

    return projected_cpu

In [24]:
model.config.use_cache = False
model.base_model.model.config._attn_implementation = "eager"

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

model.enable_input_require_grads()

model.train()

transformer = model.base_model.model.model
lm_head = model.base_model.model.lm_head

first_layer = transformer.layers[0]
checkpointing_active = (
    getattr(first_layer, "gradient_checkpointing", False)
    and first_layer.training
)

print(
    f"[checkpointing check] layer.gradient_checkpointing="
    f"{getattr(first_layer, 'gradient_checkpointing', 'MISSING')}, "
    f"layer.training={first_layer.training}, "
    f"model.training={model.training}"
)

if not checkpointing_active:
    print(
        "WARNING: gradient checkpointing will NOT engage as configured. "
        "Retrying by enabling it directly on the underlying HF model "
        "(bypassing the PeftModel wrapper) ..."
    )
    model.base_model.model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    checkpointing_active = (
        getattr(transformer.layers[0], "gradient_checkpointing", False)
        and transformer.layers[0].training
    )
    print(f"[after retry] checkpointing_active={checkpointing_active}")

if not checkpointing_active:
    raise RuntimeError(
        "Gradient checkpointing still not active on the decoder layers. "
        "Do not proceed with the full 9,000-example run -- it will OOM on "
        "example 0 exactly as before. Print `transformer.layers[0]` and check "
        "your installed `transformers`/`peft` versions for a checkpointing "
        "propagation mismatch."
    )


num_dropout_modules_disabled = 0
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.eval()
        num_dropout_modules_disabled += 1

print(f"Disabled dropout on {num_dropout_modules_disabled} module(s); "
      f"decoder layers remain in train() mode for checkpointing.")

print_gpu_memory("before production extraction")


[checkpointing check] layer.gradient_checkpointing=True, layer.training=True, model.training=True
Disabled dropout on 112 module(s); decoder layers remain in train() mode for checkpointing.
[before production extraction] allocated=3.01 GiB | reserved=3.26 GiB | peak=3.14 GiB | free=11.18 GiB / 14.56 GiB


In [25]:
import json
from pathlib import Path

def save_progress_state(state, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    clean_state = {
        Path(k).name if isinstance(k, Path) else str(k): int(v)
        for k, v in state.items()
    }

    with open(path, "w") as f:
        json.dump(clean_state, f, indent=2)


def load_progress_state(path):
    path = Path(path)

    if not path.exists():
        return {
            name: 0
            for name in CHECKPOINT_NAMES
        }

    with open(path, "r") as f:
        state = json.load(f)

    return {
        Path(k).name: int(v)
        for k, v in state.items()
    }

In [26]:
import time

CHECKPOINT_NAME = "epoch_1"
AUTO_ADVANCE = True          # if CHECKPOINT_NAME above is already finished, auto-pick the next incomplete one

PROGRESS_SAVE_INTERVAL = 50

PROGRESS_PATH = GRADIENT_DATASTORE_ROOT / "progress_state.json"

if PROGRESS_PATH.exists():
    progress_state = load_progress_state(PROGRESS_PATH)
else:
    progress_state = {
        Path(name).name: 0
        for name in checkpoint_dirs
    }

if AUTO_ADVANCE:
    _all_names = [Path(p).name for p in checkpoint_dirs]
    if progress_state.get(CHECKPOINT_NAME, 0) >= len(tokenized_candidate):
        _next_name = next(
            (
                n for n in _all_names
                if progress_state.get(n, 0) < len(tokenized_candidate)
            ),
            None,
        )
        if _next_name is not None and _next_name != CHECKPOINT_NAME:
            print(
                f"'{CHECKPOINT_NAME}' already complete -- "
                f"auto-advancing to '{_next_name}'."
            )
            CHECKPOINT_NAME = _next_name

start_idx = int(
    progress_state.get(CHECKPOINT_NAME, 0)
)

print(
    f"{CHECKPOINT_NAME}: "
    f"{start_idx:,}/{len(tokenized_candidate):,}"
)

if start_idx >= len(tokenized_candidate):
    print("✓ Already complete.")
else:

    checkpoint_path = (
        CHECKPOINT_ROOT / CHECKPOINT_NAME
    )

    model.load_adapter(
        str(checkpoint_path),
        adapter_name="default",
        is_trainable=True,
    )

    model.set_adapter("default")

    model.train()
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.eval()

    _first_layer = model.base_model.model.model.layers[0]
    _checkpointing_active = (
        getattr(_first_layer, "gradient_checkpointing", False)
        and _first_layer.training
    )
    print(
        f"[{CHECKPOINT_NAME}] checkpointing_active={_checkpointing_active}, "
        f"model.training={model.training}"
    )
    if not _checkpointing_active:
        raise RuntimeError(
            f"Gradient checkpointing not active after loading {CHECKPOINT_NAME}. "
            "Do not proceed -- this will OOM on example 0 exactly as before."
        )

    current_trainable = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    assert len(current_trainable) == 224

    projector = CudaProjector(
        grad_dim=total_gradient_dimensions,
        proj_dim=GRADIENT_DIMS,
        proj_type=PROJECTION_TYPE,
        seed=PROJECTOR_SEED,
        max_batch_size=PROJECTOR_BATCH_SIZE,
        device=device,
    )

    datastore = gradient_datastores[
        CHECKPOINT_NAME
    ]

    checkpoint_start = time.time()

    try:
        progress_bar = tqdm(
            range(
                start_idx,
                len(tokenized_candidate)
            ),
            initial=start_idx,
            total=len(tokenized_candidate),
            desc=CHECKPOINT_NAME,
            unit="example",
        )

        for example_idx in progress_bar:

            example_start = time.time()

            model.zero_grad(
                set_to_none=True
            )

            gc.collect()
            torch.cuda.empty_cache()

            example = tokenized_candidate[
                example_idx
            ]

            sequence_length = len(
                example["input_ids"]
            )

            input_ids = torch.tensor(
                example["input_ids"],
                dtype=torch.long,
                device=device,
            ).unsqueeze(0)

            attention_mask = torch.tensor(
                example["attention_mask"],
                dtype=torch.long,
                device=device,
            ).unsqueeze(0)

            labels = torch.tensor(
                example["labels"],
                dtype=torch.long,
                device=device,
            ).reshape(-1)

            with torch.autocast(
                device_type="cuda",
                dtype=MODEL_DTYPE,
            ):

                outputs = transformer(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                    return_dict=True,
                )

                hidden_states = (
                    outputs.last_hidden_state.reshape(
                        -1,
                        outputs.last_hidden_state.shape[-1],
                    )
                )

                loss = fused_loss_fn(
                    lm_head.weight,
                    hidden_states,
                    labels,
                )

            loss_value = loss.detach().item()

            loss.backward()

            for parameter in current_trainable:

                if parameter.grad is None:
                    raise RuntimeError(
                        f"Missing gradient at "
                        f"{CHECKPOINT_NAME}, "
                        f"example {example_idx}"
                    )

                if not torch.isfinite(
                    parameter.grad
                ).all():
                    raise RuntimeError(
                        f"Non-finite gradient at "
                        f"{CHECKPOINT_NAME}, "
                        f"example {example_idx}"
                    )

            adam_gradient_vector = (
                build_adam_gradient_vector(
                    model=model,
                    parameter_state_mapping=(
                        parameter_state_mapping
                    ),
                    adam_state_dict=(
                        adam_states[CHECKPOINT_NAME]
                    ),
                    device=device,
                )
            )

            with torch.no_grad():

                projected = projector.project(
                    adam_gradient_vector.unsqueeze(0),
                    model_id=0,
                ).squeeze(0)

            if projected.shape != (
                GRADIENT_DIMS,
            ):
                raise RuntimeError(
                    f"Unexpected projection shape: "
                    f"{tuple(projected.shape)}"
                )

            datastore[
                example_idx, :
            ] = (
                projected
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            progress_state[
                CHECKPOINT_NAME
            ] = example_idx + 1

            if (
                (example_idx + 1)
                % PROGRESS_SAVE_INTERVAL
                == 0
            ):
                save_progress_state(
                    progress_state,
                    PROGRESS_PATH,
                )
                datastore.flush()

            elapsed = (
                time.time()
                - example_start
            )

            progress_bar.set_postfix(
                loss=f"{loss_value:.4f}",
                length=sequence_length,
                sec=f"{elapsed:.2f}",
                gpu=f"{torch.cuda.memory_allocated() / 1024**3:.2f}G",
            )

            del input_ids
            del attention_mask
            del labels
            del outputs
            del hidden_states
            del loss
            del adam_gradient_vector
            del projected

            gc.collect()
            torch.cuda.empty_cache()

        datastore.flush()

        progress_state[
            CHECKPOINT_NAME
        ] = len(tokenized_candidate)

        save_progress_state(
            progress_state,
            PROGRESS_PATH,
        )

        elapsed = (
            time.time()
            - checkpoint_start
        )

        print(
            f"\n✓ {CHECKPOINT_NAME} complete"
        )

        print(
            f"Time: {elapsed / 3600:.2f} hours"
        )

        del projector

        model.zero_grad(
            set_to_none=True
        )

        gc.collect()
        torch.cuda.empty_cache()
    finally:
        datastore.flush()

        if "example_idx" in globals():
            progress_state[CHECKPOINT_NAME] = example_idx + 1

        save_progress_state(
            progress_state,
            PROGRESS_PATH,
        )

        print(
            f"[safety flush] {CHECKPOINT_NAME}: "
            f"progress saved through index "
            f"{progress_state.get(CHECKPOINT_NAME, start_idx):,}"
        )


'epoch_1' already complete -- auto-advancing to 'epoch_4'.
epoch_4: 0/9,000
[epoch_4] checkpointing_active=True, model.training=True


epoch_4:   0%|          | 0/9000 [00:00<?, ?example/s]


✓ epoch_4 complete
Time: 11.33 hours
[safety flush] epoch_4: progress saved through index 9,000
